In [ ]:
# The goal of this script is to plot the transfer functions for vector perturbations 
# and the associated final temperature and polarization spectra

In [ ]:
# import necessary modules
from classy import Class
from math import pi
import numpy as np
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt

In [ ]:
#####################################################
#
# Cosmological parameters and other CLASS parameters
#
#####################################################
k = .03  # 1/Mpc

common_settings = {# LambdaCDM parameters (Planck 18 + lensing + BAO bestfit)
                   'omega_b':2.255065e-02,
                   'omega_cdm':1.193524e-01,
                   'H0':6.776953e+01,
                   'A_s':2.123257e-09,
                   'k_output_values':k,
                   'z_reio':8.227371e+00,
                   'gauge':'newtonian',
                   'hierarchy':'tam'}

# Since vector modes get negligible at large l, we compute them only up to l=1000
l_max_vectors = 1000  

In [ ]:
###############
#    
# call CLASS : vectors only, with a given value of the tensor tilt n_v
#              and of the vector-to-scalar ratio r_v
#              (so A_v = r_v * A_s where A_s was passed before)
#             We also choose isocurvature initial conditions
###############
#
M1 = Class()
M1.set(common_settings) # new input
M1.set({'output':'tCl,pCl,wTk','modes':'v','lensing':'no','r_v':0.1,'n_v':0,'ic_v':'iso',
       'l_max_vectors':l_max_vectors})
clviso = M1.raw_cl(l_max_vectors)  # raw_cl gives the unlensed spectrum

M2 = Class()
M2.set(common_settings) # new input
M2.set({'output':'tCl,pCl,wTk','modes':'v','lensing':'no','r_v':0.1,'n_v':0,'ic_v':'oct',
       'l_max_vectors':l_max_vectors})
clvoct = M2.raw_cl(l_max_vectors)  # raw_cl gives the unlensed spectrum

In [ ]:
#################
#
# plotting
#
#################
#
plt.xlim([2,l_max_vectors])
plt.ylim([1.e-8,10])
plt.xlabel(r"$\ell$")
plt.ylabel(r"$\ell (\ell+1) C_l^{XY} / 2 \pi \,\,\, [\times 10^{10}]$")
plt.title(r"$r=0.1$")
plt.grid()
#
ell = clviso['ell']
factor = 1.e10*ell*(ell+1.)/2./pi
#
plt.loglog(ell,factor*clviso['tt'],'r-',label=r'$\mathrm{TT(v)}\,\mathrm{iso}$')
plt.loglog(ell,factor*clviso['ee'],'b-',label=r'$\mathrm{EE(v)}\,\mathrm{iso}$')
plt.loglog(ell,factor*clvoct['tt'],'r--',label=r'$\mathrm{TT(v)}\,\mathrm{oct}$')
plt.loglog(ell,factor*clvoct['ee'],'b--',label=r'$\mathrm{EE(v)}\,\mathrm{oct}$')
plt.legend(loc='right',bbox_to_anchor=(1.4, 0.5))
#plt.savefig('cl_V.pdf',bbox_inches='tight')

## Now the transfer functions for the vector modes

In [ ]:
# load perturbations
#
all_k = M1.get_perturbations()  # this potentially constains vectors and all k values
print (all_k['vector'][0].keys())
#    
one_k = all_k['vector'][0] 

tau = one_k['tau [Mpc]'] 
V = one_k['V (vec. mod.)']
T_1 = one_k['T_1']
T_2 = one_k['T_2']
E_2 = one_k['E_2']
B_2 = one_k['B_2']
N_1 = one_k['N_1']
N_2 = one_k['N_2']
theta_b = one_k['theta_b']

In [ ]:
plt.loglog(tau,np.abs(V),'k-',label=r'$V$')
plt.loglog(tau,np.abs(T_1),'y-',label=r'$\Theta_1$')
plt.loglog(tau,np.abs(N_1),'y--',label=r'$N_1$')

plt.loglog(tau,np.abs(theta_b),'m-',label=r'$\Theta_b$')
plt.loglog(tau,np.abs(T_2),'c-',label=r'$\Theta_2$')
plt.loglog(tau,np.abs(N_2),'c--',label=r'$N_2$')

plt.loglog(tau,np.abs(E_2),'g-',label=r'$E_2$')

plt.xlim([tau[0],2000])
plt.ylim([1e-8,10])

plt.legend(loc='right',bbox_to_anchor=(0.2, 0.3))
#plt.savefig('transfer_vectors.pdf',bbox_inches='tight')

In [ ]:
#We check that baryons and photons where indeed tight coupled 
# and that neutrinos had velocity opposite to photons in the isocurvature case
plt.semilogx(tau,np.abs(V),'k-',label=r'$V$')
plt.semilogx(tau,T_1-theta_b,'y-',label=r'$\Theta_1-\Theta_b$')
plt.semilogx(tau,T_1,'c-',label=r'$\Theta_1$')
plt.semilogx(tau,N_1,'c--',label=r'$N_1$')


plt.legend(loc='right',bbox_to_anchor=(1., 0.2))
#plt.savefig('transfer_vectors.pdf',bbox_inches='tight')